<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661-Applications-of-AI-Models/blob/main/Assignment1_661_AA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment 1: Price Prediction using Neural Networks

## 1. Data Exploration and Cleaning

In [ ]:
# Import required libraries for data analysis and visualization

import pandas as pd              # for data manipulation
import numpy as np               # for numerical operations
import matplotlib.pyplot as plt  # for plotting
import seaborn as sns            # for enhanced visualizations

# Set a default aesthetic style for plots
sns.set(style="whitegrid")

In [ ]:
# import Airbnb Listing datasets for Asheville

listings_url = 'https://data.insideairbnb.com/united-states/nc/asheville/2024-06-21/data/listings.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip')

> **Instructions covered:** Loading the "Airbnb Listings Dataset" for data exploration and cleaning. Taken directly from W3.

In [ ]:
# Ex1 a: Display the first 10 rows
listings_df.head(10)

In [ ]:
# Ex1 b: Explore columns, data types, and non-null counts
listings_df.info()

In [ ]:
# Ex1 c: Display the shape of the DataFrame (rows, columns)
listings_df.shape

> **Instructions covered:** Initial data exploration — understanding the dataset structure, columns, data types, and missing values. Taken directly from W3.

In [ ]:
# Ex 2: Print all unique host locations in the dataset
print(listings_df['host_location'].unique())      # List all location names
print(len(listings_df['host_location'].unique())) # Total number of unique locations

In [ ]:
# Ex 3: Filter Listings for Asheville
asheville_df = listings_df[listings_df['host_location'] == 'Asheville, NC']

In [ ]:
# Shape of the filtered dataset
asheville_df.shape

> **Instructions covered:** "Ensure listings belong to hosts located in Asheville, NC" — "Keep only records where host_location is `Asheville, NC`. Drop rows with missing or different host_location values." Taken directly from W3.

In [ ]:
#Ex 4:  Define the columns we want to keep and create a new dataframe with the selected columns
selected_columns = [
    'price', 'bathrooms','bedrooms', 'number_of_reviews',
    'room_type', 'host_identity_verified', 'host_is_superhost',
    'accommodates', 'review_scores_rating'
]

asheville_df = asheville_df[selected_columns]

In [ ]:
# Preview the first few rows of the new dataframe
asheville_df.head()

> **Instructions covered:** "Extend the in-class features (bedrooms, bathrooms, number_of_reviews, host_is_superhost, host_identity_verified) by adding: accommodates, review_scores_rating, and room_type." Column selection extended from W3 to include `accommodates` and `review_scores_rating`. Also removed `latitude` and `longitude` as they are not required by the assignment.

In [ ]:
# 5.1: Check for missing values in each column
asheville_df.isnull().sum()

> **Instructions covered:** Checking for missing values before applying imputation strategies. Taken directly from W3.

In [ ]:
# 5.2: Drop rows where the target variable (price) is missing and then re-check for missing values
asheville_df = asheville_df.dropna(subset=['price'])
asheville_df.isnull().sum()

> **Instructions covered:** "Drop rows where the price value is missing." Taken directly from W3.

In [ ]:
# 5.3: Use one of the imputation methods to impute missing values, if there are any missing values after dropping records in step 2
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].fillna(
    asheville_df['host_is_superhost'].mode()[0]
)
asheville_df.isnull().sum()

> **Instructions covered:** "Replace categorical missing values with mode." — Imputes `host_is_superhost` using mode. Taken directly from W3.

In [ ]:
# 5.4: Impute numerical missing values using mean or median, depending on data distribution
asheville_df['bedrooms'] = asheville_df['bedrooms'].fillna(asheville_df['bedrooms'].median())
asheville_df['bathrooms'] = asheville_df['bathrooms'].fillna(asheville_df['bathrooms'].mean())
asheville_df.isnull().sum()

> **Instructions covered:** "For numerical features (except review_scores_rating), fill missing values using the mean or median, depending on data distribution." — Imputes `bedrooms` using median and `bathrooms` using mean.

In [ ]:
# 5.5: For review_scores_rating, replace missing values with 0 (indicating no reviews)
asheville_df['review_scores_rating'] = asheville_df['review_scores_rating'].fillna(0)

# Create a dummy variable (has_review_scores) to indicate whether a listing has a review
asheville_df['has_review_scores'] = (asheville_df['review_scores_rating'] > 0).astype(int)

asheville_df.isnull().sum()

> **Instructions covered:** "For review_scores_rating, replace missing values with 0 (indicating no reviews). Create a dummy variable (has_review_scores) to indicate whether a listing has a review: Set has_review_scores = 1 if review_scores_rating > 0. Set has_review_scores = 0 if review_scores_rating == 0 (no reviews)."

In [ ]:
# check data types
asheville_df.dtypes

In [ ]:
# 6.2: Convert price to Numeric
asheville_df['price'] = asheville_df['price'].replace(r'[\$,]', '', regex=True).astype(float)

> **Instructions covered:** Correcting data types — converting `price` from string (object) to numeric for analysis. Taken directly from W3.

In [ ]:
# 6.3: Convert Boolean-like Columns
asheville_df['host_identity_verified'] = asheville_df['host_identity_verified'].map({'t': 1, 'f': 0})
asheville_df['host_is_superhost'] = asheville_df['host_is_superhost'].map({'t': 1, 'f': 0})

> **Instructions covered:** "Encode categorical variables with... dummy coding (binary)" — converting `host_identity_verified` and `host_is_superhost` from 't'/'f' strings to 1/0. Taken directly from W3.

In [ ]:
# 6.4: Encode Categorical Column

room_type_dummies = pd.get_dummies(asheville_df['room_type'], prefix='room_type', drop_first=True)
room_type_dummies = room_type_dummies.astype(int)

# Add new columns to the DataFrame
asheville_df = pd.concat([asheville_df, room_type_dummies], axis=1)
asheville_df.drop('room_type', axis=1, inplace=True)

asheville_df.head()

> **Instructions covered:** "Encode categorical variables with one-hot encoding (multi-category)" — one-hot encoding `room_type`. Taken directly from W3.

In [ ]:
# Visualize the original price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (Before Removing Top 1%)")
plt.xlabel("Price")
plt.show()

In [ ]:
# Remove top 1% of extreme price values
price_cap = asheville_df['price'].quantile(0.99)
asheville_df = asheville_df[asheville_df['price'] <= price_cap]

In [ ]:
# Visualize the cleaned price distribution
plt.figure(figsize=(10, 4))
sns.histplot(asheville_df['price'], bins=50, kde=True, color='green')
plt.title("Price Distribution (After Removing Top 1%)")
plt.xlabel("Price")
plt.show()

> **Instructions covered:** "Handle outliers (e.g., apply the 99th percentile rule to filter out extreme price values)." Taken directly from W3.

In [ ]:
# 7.1: Descriptive Statistics

asheville_df.describe()

> **Instructions covered:** "Provide descriptive statistics (i.e., min, max, mean, and standard deviation) of all numerical variables." Taken directly from W3.

### Discussion and Reflection

> **Instructions covered:** "Briefly discuss descriptive statistics and any insights gained from the data exploration. What additional data cleaning or preprocessing techniques could you have applied to further improve the model's performance?"

*TODO: Write discussion here after running the notebook.*

## 2. Exploratory Data Analysis

In [ ]:
# 7.2: Histograms of Distributions

numeric_cols = ['price', 'bedrooms', 'bathrooms', 'number_of_reviews', 'accommodates', 'review_scores_rating']
asheville_df[numeric_cols].hist(bins=30, figsize=(12,10))
plt.tight_layout()
plt.show()

> **Instructions covered:** "Plot histograms to visualize variable distributions." Updated from W3 to include the new features `accommodates` and `review_scores_rating`.

In [ ]:
corr = asheville_df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

> **Instructions covered:** "Calculate and visualize correlations between the variables." Taken directly from W3.

### Discussion

> **Instructions covered:** "Discuss your findings and any insights drawn from variable distributions and correlation matrices."

*TODO: Write discussion here after running the notebook.*

## 3. Model Building and Evaluation

In [ ]:
# Imports required libraries for pre-processing and modeling

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Normalization, Input
from sklearn.model_selection import train_test_split

# Selecting a subset of numeric features relevant to pricing.
features = ['bedrooms', 'bathrooms', 'number_of_reviews', 'host_is_superhost', 'host_identity_verified',
            'accommodates', 'review_scores_rating', 'has_review_scores',
            'room_type_Hotel room', 'room_type_Private room', 'room_type_Shared room']
target = 'price'

# Define X and y
X = asheville_df[features]
y = asheville_df[target]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define and Adapt Normalization Layer
norm_layer = Normalization()
norm_layer.adapt(X_train.values)

> **Instructions covered:** "Split the data into training (80%) and testing (20%) sets." — "Normalize features before training." Updated from W3 to include all assignment features: `accommodates`, `review_scores_rating`, `has_review_scores`, and the `room_type` dummies.

In [ ]:
# Build the Sequential Model
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    norm_layer,
    Dense(64, activation='relu'),
    Dense(1)  # Regression output
])

> **Instructions covered:** "Build a neural network regression model using Keras Sequential" — "Input: one node per feature." — "At least one hidden layer with 'relu' activation" — "Output: one node (default linear activation) for price." Taken directly from W3.

In [ ]:
# Compile the Model
model.compile(optimizer='adam', loss='mse', metrics=['mae', 'mse'])

> **Instructions covered:** "Compile the model with optimizer = 'adam' and loss = 'mse'. Track metrics such as 'mae' and 'mse'." Updated from W3 to also track `'mse'`.

In [ ]:
model.summary()

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    validation_split=.2,
    epochs=50,
    verbose= 1
)

> **Instructions covered:** "Train the model for a reasonable number of epochs (e.g., 50–100) with validation split of 20%." Taken directly from W3.

In [ ]:
import matplotlib.pyplot as plt

# Plot MAE over epochs
plt.plot(history.history['mae'], label='Train MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('MAE (Price)')
plt.title('Training vs Validation MAE')
plt.legend()
plt.grid(True)
plt.show()

# Plot Loss (MSE) over epochs
plt.plot(history.history['loss'], label='Train Loss (MSE)')
plt.plot(history.history['val_loss'], label='Validation Loss (MSE)')
plt.xlabel('Epochs')
plt.ylabel('MSE')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

> **Instructions covered:** "Evaluate model performance using... training/validation loss curves." Taken directly from W3.

In [ ]:
# Evaluate on Test Set
test_loss, test_mae, test_mse = model.evaluate(X_test, y_test, verbose=0)
print(f"Test MAE: ${test_mae:.2f}")
print(f"Test MSE: {test_mse:.2f}")
print(f"Test Loss: {test_loss:.2f}")

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

# Predict on the test set
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"R² Score: {r2:.2f}")

> **Instructions covered:** "Evaluate model performance using test set MSE, R²." Taken directly from W3.

### Discussion

> **Instructions covered:** "Did the model overfit or underfit? Discuss your findings and suggest improvements for better performance."

*TODO: Write discussion here after running the notebook.*